In [3]:


# 1. Import Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import joblib


# ================================
# 2. Load Processed Dataset
# ================================

X = pd.read_csv("../model/X_scaled.csv")
y = pd.read_csv("../model/y.csv")

y = y.values.ravel()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


# ================================
# 3. Train Test Split
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Test Data:", X_test.shape)


# ================================
# 4. Cross Validation Setup
# ================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ================================
# 5. Logistic Regression Model
# ================================

print("\nTraining Logistic Regression...")

log_model = LogisticRegression(max_iter=1000)

log_params = {
    "C": [0.01, 0.1, 1, 10]
}

log_grid = GridSearchCV(
    log_model,
    log_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

log_grid.fit(X_train, y_train)

best_log_model = log_grid.best_estimator_

print("Best Logistic Parameters:", log_grid.best_params_)


# ================================
# 6. Random Forest Model
# ================================

print("\nTraining Random Forest...")

rf_model = RandomForestClassifier(random_state=42)

rf_params = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    rf_model,
    rf_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf_model = rf_grid.best_estimator_

print("Best RF Parameters:", rf_grid.best_params_)


# ================================
# 7. XGBoost Model
# ================================

print("\nTraining XGBoost...")

xgb_model = XGBClassifier(
    eval_metric="logloss",
    use_label_encoder=False
)

xgb_params = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.01, 0.1]
}

xgb_grid = GridSearchCV(
    xgb_model,
    xgb_params,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

xgb_grid.fit(X_train, y_train)

best_xgb_model = xgb_grid.best_estimator_

print("Best XGB Parameters:", xgb_grid.best_params_)


# ================================
# 8. Model Evaluation Function
# ================================

def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("ROC AUC:", roc_auc_score(y_test, y_prob))

    print("\nClassification Report\n")
    print(classification_report(y_test, y_pred))


# ================================
# 9. Evaluate Models
# ================================

print("\nLogistic Regression Performance")
evaluate_model(best_log_model, X_test, y_test)

print("\nRandom Forest Performance")
evaluate_model(best_rf_model, X_test, y_test)

print("\nXGBoost Performance")
evaluate_model(best_xgb_model, X_test, y_test)


# ================================
# 10. Feature Importance
# ================================

print("\nTop Important Features")

feature_importance = pd.Series(
    best_rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance.head(10))


# ================================
# 11. Save Best Model
# ================================

print("\nSaving Model...")

joblib.dump(best_xgb_model, "../model/churn_model.pkl")

print("Model saved to model/churn_model.pkl")


# ================================
# 12. Load Model (for deployment)
# ================================

model = joblib.load("../model/churn_model.pkl")

print("Model loaded successfully")

Feature shape: (7043, 30)
Target shape: (7043,)
Training Data: (5634, 30)
Test Data: (1409, 30)

Training Logistic Regression...
Best Logistic Parameters: {'C': 10}

Training Random Forest...
Best RF Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}

Training XGBoost...


c:\Users\kiara\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [12:09:42] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best XGB Parameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}

Logistic Regression Performance
Accuracy: 0.7970191625266146
Precision: 0.6309523809523809
Recall: 0.5668449197860963
F1 Score: 0.5971830985915493
ROC AUC: 0.8481050918391071

Classification Report

              precision    recall  f1-score   support

           0       0.85      0.88      0.86      1035
           1       0.63      0.57      0.60       374

    accuracy                           0.80      1409
   macro avg       0.74      0.72      0.73      1409
weighted avg       0.79      0.80      0.79      1409


Random Forest Performance
Accuracy: 0.8048261178140526
Precision: 0.6633663366336634
Recall: 0.5374331550802139
F1 Score: 0.5937961595273265
ROC AUC: 0.8517089049058358

Classification Report

              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1035
           1       0.66      0.54      0.59       374

    accuracy                      